# Data preparation of outage data from ENTSO-E transparency

source: https://transparency.entsoe.eu/

Creates the following parsed datasets

- Monthly availability for peakload technlogies and country for selected year - both on a generation and production unit level (avail_peakload_"+outagetype+"_"+year+"_monthly_entsoe.csv)

Settings in next window

In [1]:
#download files again (yes/no)?
download = "no"

#set year for data creation
year = '2017'

#outage type GU/PU
outagetype = "GU"

In [2]:
import pysftp
import sys
import os
import pandas as pd
import datetime as dt
import wget
import calendar

In [3]:
#ENTSO-E sftp settings
host = "sftp-transparency.entsoe.eu"
password = "qhfWzbuxRkmmKb+"                
username = "jonas.savelsberg@unibas.ch"                
port = '22'

In [4]:
dir_out = "../parsed_data/"

In [5]:
#technology definition
dict_agg_tech = {'Fossil Oil shale ': 'Oil',
                 'Fossil Gas ': 'Gas',
                 'Fossil Brown coal/Lignite ': 'Lignite',
                 'Hydro Water Reservoir ': 'Reservoir',
                 'Hydro Pumped Storage ': 'Pump',
                 'Other ': 'Other',
                 'Fossil Hard coal ': 'HardCoal',
                 'Fossil Oil ': 'Oil',
                 'Nuclear ': 'Nuclear',
                 'Wind Offshore ': 'WindOffshore',
                 'Biomass ': 'Biomass',
                 'Wind Onshore ': 'WindOnshore',
                 'Hydro Run-of-river and poundage ': 'RunOfRiver'}

In [6]:
dict_agg_country = {
    'EE': 'EE',
    'HU': 'HU',
    'CZ': 'CZ',
    'BE': 'BE',
    'CH': 'CH',
    'DE_TransnetBW': 'DE',
    'DE_AT_LU': 'DE',
    'FR': 'FR',
    'NO2': 'NO',
    'NO': 'NO',
    'DE_Amprion': 'DE',
    'NO5': 'NO',
    'IT_CNOR': 'IT',
    'IT_North': 'IT',
    'IT_SARD': 'IT',
    'IT_PRGP': 'IT',
    'IT_BRNN': 'IT',
    'IT_CSUD': 'IT',
    'IT_SICI': 'IT',
    'IT_FOGN': 'IT',
    'IT_ROSN': 'IT',
    'IT': 'IT',
    'SE2': 'SE',
    'SE': 'SE',
    'DK2': 'DK',
    'ES': 'ES',
    'DK': 'DK',
    'NO1': 'NO',
    'SE1': 'SE',
    'SE3': 'SE',
    'RO': 'RO',
    'LV': 'LV',
    'DE_TenneT_GER': 'DE',
    'AL': 'AL',
    'PL': 'PL',
    'GB': 'GB',
    'FI': 'FI',
    'LT': 'LT',
    'DK1': 'DK',
    'NO3': 'NO',
    'NO4': 'NO',
    'DE_50HzT': 'DE',
    'SE4': 'SE',
    'MK': 'MK',
    'AT': 'AT',
    'NL': 'NL',
    'SK': 'SK'}

In [7]:
months = pd.DataFrame()
for m in range(1,13):
    months[m] = calendar.monthrange(int(year), m)
dayspermonth = months.loc[1]

In [8]:
#Connect to ENTSO-E Transparency FTP
#for this to work, pysftp 0.2.8 is needed!
path = '/TP_export/'
path_local = os.path.join(os.pardir,'source_data/ENTSOE/') 

In [9]:
# show list of all available folders (uncomment last line if needed)
with pysftp.Connection(host=host, username=username, password=password) as sftp:
    print("Connection succesfully established.")
    files = sftp.listdir('/TP_export/')   
    #print(files)

Connection succesfully established.


In [10]:
#load file names from server
path_out = path+'Outages'+outagetype+'/'
path_out_local = path_local+'outages/'
with pysftp.Connection(host=host, username=username, password=password) as sftp:
    print("Connection succesfully established.")
    # show list of files
    files = sftp.listdir(path_out)
    if year != "":
        files = [i for i in files if year in i]

Connection succesfully established.


In [11]:
#download aggregated generation data (AggregatedGenerationPerType)
if download == "yes":
    with pysftp.Connection(host=host, username=username, password=password) as sftp:
        for file in files:
            sftp.get(path_out+file,path_out_local+file)
            print('Successfully downloaded file '+file)

In [12]:
#combine files to one data frame
df_out_in = pd.DataFrame()
for file in files:
    df_temp = pd.read_csv(path_out_local+file,
                          decimal=".",encoding="UTF-16LE",sep="\t")
    df_out_in = df_out_in.append(df_temp)
df_out_in = df_out_in[(df_out_in.Status == "Active") 
                          & (df_out_in.Type == "Planned") 
                          #& (df_out_in.UnavailabilityValue > 0)
                          & (df_out_in.AreaTypeCode == "CTA")
                         ]
df_out_in["technology"] = df_out_in.ProductionType.map(dict_agg_tech)
df_out_in["country"] = df_out_in.MapCode.map(dict_agg_country)
#we ignore timezones here as we are only interested in monthly values
df_out_in['duration'] = (pd.to_datetime(df_out_in['EndTS']) - pd.to_datetime(df_out_in['StartTS']))/ pd.Timedelta('1 hour')
df_out_in['gen_reduction'] = df_out_in['UnavailabilityValue'] * df_out_in['duration']
df_out_in['dayspermonth'] = df_out_in.Month.map(dayspermonth)
    
if outagetype == "PU":
    df_out_in['fullload'] = df_out_in['dayspermonth'] * 24 * df_out_in['InstalledCapacity']
    
if outagetype == "GU":
    df_out_in['fullload'] = df_out_in['dayspermonth'] * 24 * df_out_in['InstalledGenCapacity']

df_out_in.head()

,Year,Month,Day,StartTS,EndTS,TimeZone,MRID,Status,Type,areacode,...,UnavailabilityValue,Version,Reason,UpdateTime,technology,country,duration,gen_reduction,dayspermonth,fullload
18,2017,10,14,2017-10-14 00:00:00.000,2017-10-17 00:00:00.000,WET,kr66KnnSjwO9CpxbTpIykQ,Active,Planned,10YGB----------A,...,0,1,Foreseen Maintenance,2018-10-02 11:42:14,Gas,GB,72.0,0.0,31,372000.0
20,2017,10,7,2017-10-07 00:00:00.000,2017-10-10 00:00:00.000,WET,APqdEN17lLucTjPuJkwgcg,Active,Planned,10YGB----------A,...,0,1,Foreseen Maintenance,2018-10-02 11:42:14,Gas,GB,72.0,0.0,31,372000.0
36,2017,10,6,2017-10-06 22:00:00.000,2017-10-10 22:00:00.000,WET,sNsLJRtkQFC0I8A_gi8cKw,Active,Planned,10YGB----------A,...,0,1,Foreseen Maintenance,2018-10-02 11:42:17,HardCoal,GB,96.0,0.0,31,360840.0
48,2017,10,6,2017-10-06 22:00:00.000,2017-10-17 14:00:00.000,WET,htrpYsfV6JwD4tqkfJhgvg,Active,Planned,10YGB----------A,...,0,1,Foreseen Maintenance,2018-10-02 11:42:21,HardCoal,GB,256.0,0.0,31,360840.0
887,2017,10,9,2017-10-09 01:00:00.000,2017-10-16 00:30:00.000,CET,7NkVM79YCcwerNyUtsTxRw,Active,Planned,10Y1001A1001A016,...,0,1,Foreseen Maintenance,2018-10-02 11:51:38,Oil,NaN,167.5,0.0,31,43152.0


In [13]:
if outagetype == "PU":
    df_out = df_out_in.groupby(['Month','country','technology']).sum()[['InstalledCapacity','fullload','gen_reduction']]
    
if outagetype == "GU":
    df_out = df_out_in.groupby(['Month','country','technology']).sum()[['InstalledGenCapacity','fullload','gen_reduction']]

df_out['availability'] = (df_out['fullload'] - df_out['gen_reduction']) / df_out['fullload']
df_out.head()

InstalledGenCapacity    fullload  gen_reduction  \
Month country technology                                                    
1     AT      Gas                       3732.4   2776905.6           0.00   
              HardCoal                   335.0    249240.0           3.75   
      BE      Gas                       2018.8   1501987.2          29.00   
      CH      Nuclear                   1220.0    907680.0           0.00   
              Pump                     31058.0  23107152.0      907127.00   

                          availability  
Month country technology                
1     AT      Gas             1.000000  
              HardCoal        0.999985  
      BE      Gas             0.999981  
      CH      Nuclear         1.000000  
              Pump            0.960743

In [14]:
df_availability = df_out['availability'].reset_index().pivot_table(values='availability', index=['country','technology'], columns='Month')
df_availability = df_availability.fillna(1)
df_availability = df_availability.reset_index()[df_availability.reset_index().technology.isin(['Gas','Oil','HardCoal'])].set_index(['country','technology'])
df_availability = abs(df_availability)
df_availability[df_availability > 1] = 1
df_availability = pd.DataFrame(df_availability.stack()).rename(columns = {0:'avail'})
df_availability.head()

avail
country technology Month          
AT      Gas        1      1.000000
                   2      0.995858
                   3      0.997095
                   4      0.999850
                   5      1.000000

In [15]:
df_availability.to_csv(dir_out + "avail_peakload_"+outagetype+"_"+year+"_monthly_entsoe.csv", index=True)